# LLM Gateway Lab — LiteLLM + LangChain

A practical lab designed to lock in the **entire LLM Gateway concept**.

We will build and understand:

- Gateway abstraction
- Logical model names vs provider model IDs
- DeepSeek as primary
- Gemini 3.6 Flash as backup
- Explicit fallback
- Retry vs fallback
- Load balancing
- LangChain → LiteLLM
- Gateway authentication
- Rate limiting
- Cost/budget control
- Logging/observability
- Caching
- Guardrails
- Failure testing

Core architecture:

```text
LangChain / LangGraph / RAG / Agents
                  |
                  v
            LiteLLM Proxy
             LLM Gateway
                  |
          +-------+-------+
          |               |
       DeepSeek       Gemini 3.6 Flash
       primary          fallback
```

**Important:** Put API keys in `.env`, never directly in source/config committed to Git.


## 0. Current model choice

This lab uses:

- `deepseek/deepseek-chat`
- `gemini/gemini-3.6-flash`

Gemini's current documentation lists `gemini-3.6-flash` as a stable model with no announced shutdown date. Older Gemini 2.x Flash models have already been shut down.

We also intentionally do **not** configure Gemini `temperature`, `top_p`, or `top_k`: current Gemini documentation says these sampling parameters are deprecated for Gemini 3.6 Flash and future generations.


In [ ]:
# Install once in your project virtual environment.
# Run from the terminal if you prefer:
#
# pip install "litellm[proxy]" langchain-openai python-dotenv openai requests
#
# Or inside Jupyter:
# %pip install "litellm[proxy]" langchain-openai python-dotenv openai requests


## 1. `.env`

Create a `.env` file in the project root:

```env
DEEPSEEK_API_KEY=your_deepseek_api_key
GOOGLE_API_KEY=your_google_api_key
LITELLM_MASTER_KEY=sk-your-local-gateway-key
```

Why?

```text
Application
     |
     | gateway key
     v
LiteLLM Gateway
     |
     | provider keys
     v
DeepSeek / Gemini
```

The application should not need to contain the provider secrets.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

for name in ["DEEPSEEK_API_KEY", "GOOGLE_API_KEY"]:
    print(name, "FOUND" if os.getenv(name) else "MISSING")


DEEPSEEK_API_KEY FOUND
GOOGLE_API_KEY FOUND


## 2. The key abstraction: logical model vs real provider model

Your application asks for:

```text
reasoning
```

The gateway maps that logical name to an actual provider deployment.

```text
reasoning
    |
    +--> deepseek/deepseek-chat
```

This is why your agents/RAG application does not need to hard-code provider-specific model IDs everywhere.


## 3. Build a deterministic primary → backup fallback

For learning fallback clearly, use **different logical names**:

- `reasoning` = primary
- `reasoning_backup` = backup

Then explicitly configure:

```text
reasoning -> reasoning_backup
```

This is better for learning than assuming the second YAML item automatically means "backup".

LiteLLM Router supports retry/fallback behavior across deployments.


In [ ]:
from pathlib import Path

gateway_dir = Path("gateway_lab")
gateway_dir.mkdir(exist_ok=True)

fallback_config = r'''
model_list:

  - model_name: reasoning
    litellm_params:
      model: deepseek/deepseek-chat
      api_key: os.environ/DEEPSEEK_API_KEY

  - model_name: reasoning_backup
    litellm_params:
      model: gemini/gemini-3.6-flash
      api_key: os.environ/GOOGLE_API_KEY

router_settings:
  num_retries: 1
  timeout: 30
  fallbacks:
    - reasoning:
        - reasoning_backup

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
'''.strip()

path = gateway_dir / "config_fallback.yaml"
path.write_text(fallback_config)

print(path.resolve())
print()
print(fallback_config)


## 4. Start the gateway

From a terminal:

```bash
litellm --config gateway_lab/config_fallback.yaml --port 4000
```

Keep that terminal running.

You have now created a real local gateway:

```text
http://localhost:4000
```

The application no longer needs to contact DeepSeek directly.


In [ ]:
import requests

try:
    r = requests.get("http://localhost:4000/health", timeout=5)
    print("Gateway HTTP status:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("Gateway is not reachable.")
    print("Start it with:")
    print("litellm --config gateway_lab/config_fallback.yaml --port 4000")
    print("Error:", e)


## 5. Test the gateway WITHOUT LangChain first

This is an important engineering habit.

Test bottom-up:

```text
Provider
   ^
Gateway
   ^
LangChain
   ^
RAG / Agent
```

If the gateway itself is broken, adding LangChain makes debugging harder.


In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="http://localhost:4000",
    api_key=os.getenv("LITELLM_MASTER_KEY", "local-dev-key")
)

response = client.chat.completions.create(
    model="reasoning",
    messages=[
        {"role": "user", "content": "Explain RAG in one short sentence."}
    ]
)

print(response.choices[0].message.content)


## 6. Now connect LangChain

LangChain becomes a **client of the gateway**.

It does not need to know:

- DeepSeek API URL
- Gemini API URL
- provider credentials
- fallback rules

It only knows the gateway endpoint and logical model name.


In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="reasoning",
    api_key=os.getenv("LITELLM_MASTER_KEY", "local-dev-key"),
    base_url="http://localhost:4000"
)

result = llm.invoke(
    "Explain contextual recall in one sentence."
)

print(result.content)


## 7. The experiment that locks in fallback

Temporarily make the DeepSeek key invalid:

```env
DEEPSEEK_API_KEY=invalid_key_for_testing
```

Restart the LiteLLM proxy.

Then run the LangChain cell again.

Expected flow:

```text
LangChain
   |
   v
LiteLLM
   |
   v
DeepSeek
   |
   X failure
   |
   v
reasoning_backup
   |
   v
Gemini 3.6 Flash
   |
   v
response
```

Restore the real DeepSeek key afterward.

This demonstrates **failover**.

It is not the same thing as load balancing.


## 8. Retry vs fallback

These are different.

### Retry

> Try the same deployment again.

```text
DeepSeek
   |
   X
   |
 retry
   |
   X
```

### Fallback

> Move to another configured deployment.

```text
DeepSeek
   |
   X
   |
   v
Gemini
```

A production setup can use both:

```text
Primary
  |
 failure
  |
 retry
  |
 failure
  |
 fallback
  |
 Backup
```


## 9. Load balancing

Load balancing is different from fallback.

Instead of:

```text
DeepSeek -> failure -> Gemini
```

you may have:

```text
reasoning_pool
      |
      +--> deployment A
      +--> deployment B
      +--> deployment C
```

The goal is to distribute traffic across healthy deployments.

For example, LiteLLM's Router supports routing strategies such as `simple-shuffle`.


In [ ]:
load_balance_config = r'''
model_list:

  - model_name: reasoning_pool
    litellm_params:
      model: deepseek/deepseek-chat
      api_key: os.environ/DEEPSEEK_API_KEY

  - model_name: reasoning_pool
    litellm_params:
      model: deepseek/deepseek-chat
      api_key: os.environ/DEEPSEEK_API_KEY

router_settings:
  routing_strategy: simple-shuffle
  num_retries: 1

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
'''.strip()

path = gateway_dir / "config_load_balancing.yaml"
path.write_text(load_balance_config)

print(path.resolve())
print()
print(load_balance_config)


### Memorize the distinction

**Fallback:**

```text
A fails -> B
```

**Load balancing:**

```text
A
B  <- traffic distributed
C
```

**Retry:**

```text
A fails -> try A again
```

Interviewers love this distinction.


## 10. Authentication

A gateway can become the centralized authentication boundary:

```text
Application
    |
    | gateway key
    v
LiteLLM
    |
    | provider key
    v
LLM provider
```

For the local lab, `LITELLM_MASTER_KEY` demonstrates the idea.

In production, use proper key management, authorization, rotation and least privilege.


## 11. Rate limiting

Suppose dozens of agents share the same provider.

Without centralized control:

```text
Agents ---> provider
Agents ---> provider
Agents ---> provider
```

With a gateway:

```text
Agents
   |
   v
Gateway
   |
   +--> rate limits
   +--> budgets
   +--> authorization
   |
   v
Provider
```

This is one reason organizations use gateways.


## 12. Cost and budgets

A gateway can centralize:

- token usage
- spend tracking
- project/user budgets
- model-level cost monitoring

Conceptually:

```text
Agents
   |
   v
Gateway
   |
   +--> usage
   +--> cost
   +--> budget
   |
   v
Models
```

This is much easier to govern centrally than implementing separate accounting in every agent.


## 13. Logging and observability

Gateway traffic can become another observability layer:

```text
Application
    |
    v
Gateway
    |
    +--> request logs
    +--> provider
    +--> latency
    +--> errors
    +--> token usage
    |
    v
LLM
```

This complements application-level tracing.

For example:

- **LangSmith:** useful for LangChain/LangGraph application traces.
- **Gateway observability:** useful for centralized model traffic.


## 14. Caching

A gateway can also be a useful centralized cache point:

```text
request
  |
  v
gateway
  |
  +--> cache hit ---> response
  |
  +--> cache miss --> model
```

Caching is an optimization, not the fundamental definition of a gateway.


## 15. Guardrails

A gateway can act as a centralized enforcement point:

```text
Application
    |
    v
Gateway
    |
    +--> input policy
    +--> routing
    +--> output policy
    |
    v
LLM
```

A dedicated guardrails framework can provide the actual policy engine.

Think:

- **Gateway = central traffic/control point**
- **Guardrails = policy enforcement mechanism**


## 16. Full architecture

```text
                  NEXUS AI / RAG / AGENTS
                           |
                           v
                    LangChain / LangGraph
                           |
                           v
                 +----------------------+
                 |     LiteLLM Proxy    |
                 |     LLM Gateway      |
                 +----------------------+
                    |      |       |
                 routing  auth   budgets
                    |      |       |
                    +------+-------+
                           |
                 Provider deployments
                    /       |       \
                   v        v        v
               DeepSeek   Gemini    Other
```

The gateway is **infrastructure**, not another agent.

LangGraph orchestrates.
LangChain provides application/model abstractions.
LiteLLM sits between applications and providers.
Providers perform inference.


# Interview cheat sheet

### What is an LLM Gateway?

> A centralized service between AI applications and model providers that provides a unified interface and centralizes concerns such as routing, retries, fallbacks, authentication, rate limiting, cost tracking and observability.

### Why use one?

> To decouple applications from individual providers and centralize operational and governance concerns.

### Fallback vs retry?

> Retry means trying the same deployment again. Fallback means moving to another configured deployment.

### Fallback vs load balancing?

> Fallback is failure handling. Load balancing distributes traffic across deployments.

### Does YAML order automatically mean primary then backup?

> No. Routing/fallback behavior should be explicitly configured. Multiple deployments can be used for load balancing or failover depending on Router configuration.

### Where does LangChain fit?

> LangChain is on the application side. It can call the gateway through an OpenAI-compatible endpoint, while LiteLLM handles provider-level routing and gateway concerns.


# Final mental model

Remember this:

```text
              YOUR AI APPLICATION
                      |
                      v
                LLM GATEWAY
                  LiteLLM
                      |
          +-----------+-----------+
          |           |           |
       Routing     Fallback    Governance
          |           |           |
          +-----------+-----------+
                      |
             +--------+--------+
             |        |        |
             v        v        v
         DeepSeek  Gemini    Other
```

**Your application asks WHAT model role it wants.**

```text
model="reasoning"
```

**The gateway decides HOW/WHERE to serve it.**

```text
routing
retry
fallback
auth
rate limits
cost
logging
caching
policies
```

**The provider performs the actual inference.**
